In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import torchaudio
import torchvision.models as models

# 警告を無視
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_spectrogram_efficientnet.csv'

# ハイパーパラメータ
MAX_LEN = 40          
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 20           # EfficientNetは学習が早いので20で十分
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 前処理 (Time-Weighting継承)
# ==============================================================================
def apply_time_weighting(df):
    df_eng = df.copy()
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    brain_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    max_day = df_eng['day_n'].max()
    day_norm = df_eng['day_n'] / max_day
    gain_factor = 1.0 + (day_norm * 0.5)
    
    for col in brain_cols:
        df_eng[f'{col}_weighted'] = df_eng[col] * gain_factor
    return df_eng

print(">>> Loading Data...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 前処理適用
train = apply_time_weighting(train)
test = apply_time_weighting(test)

target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

# 標準化
scaler = StandardScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset (Spectrogram変換用)
# ==============================================================================
class SpecDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        # [Time, Features]
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        # Padding
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        # Mask
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {'x': torch.tensor(x_pad), 'mask': torch.tensor(mask), 'id': sample_id}
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
            
        return ret

# ==============================================================================
# 4. Model: EfficientNet with Mel-Spectrogram
# ==============================================================================
class EfficientSpecModel(nn.Module):
    def __init__(self, input_channels, hidden_dim=128):
        super().__init__()
        
        # --- 1. Spectrogram変換層 (GPU上で高速計算) ---
        # n_fftを小さく設定しているのは、MAX_LEN=40と短いためです。
        # 時間解像度を維持しつつ周波数成分を取り出します。
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=100,  # 仮のサンプリングレート
            n_fft=32,         # ウィンドウサイズ
            hop_length=1,     # 1ステップずつずらす（時間解像度維持）
            n_mels=64         # 周波数ビンの数（画像の高さになる）
        )
        
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        
        # --- 2. Backbone (EfficientNet-B0) ---
        # 転移学習ではなく、構造だけ借りて重みはスクラッチ学習でも十分精度が出ます
        # (今回のデータは自然画像とは性質が違うため)
        self.backbone = models.efficientnet_b0(pretrained=True)
        
        # 入力チャンネル数の調整
        # EfficientNetは通常RGB(3ch)ですが、今回は脳波チャンネル数(Feature数)を入力とみなすか、
        # あるいは1chの画像をFeature数分重ねるか。
        # ここでは「入力チャネル = 1 (グレースケール)」になるようにConv層を調整します。
        # スペクトログラムの出力: [Batch, Channel(Features), n_mels, Time]
        # これを画像として扱います。
        
        # 入力: [Batch, Features, n_mels, Time] -> Conv2dで処理
        # しかしFeaturesが多い(数百)と重すぎるので、最初に畳み込んで3chにします
        self.feature_compress = nn.Conv2d(input_channels, 3, kernel_size=1)
        
        # 出力層の調整
        self.backbone.classifier[1] = nn.Linear(self.backbone.classifier[1].in_features, 1)

    def forward(self, x):
        # x: [Batch, Time, Features] -> [Batch, Features, Time]
        x = x.permute(0, 2, 1)
        
        # --- Spectrogram生成 ---
        # Input to MelSpec: [Batch, Features, Time]
        # Output: [Batch, Features, n_mels, Time]
        spec = self.mel_transform(x)
        spec = self.amplitude_to_db(spec)
        
        # 画像として扱うための正規化 (Min-Max的な処理)
        # 対数スケールなので、大体 -80 ~ 0 くらいの範囲
        spec = (spec + 80.0) / 80.0 
        
        # --- CNN処理 ---
        # [Batch, Features, H, W] -> [Batch, 3, H, W]
        img = self.feature_compress(spec)
        
        # EfficientNetへ
        out = self.backbone(img)
        
        # [Batch, 1] -> [Batch]
        # EfficientNetは画像のクラス(1つの値)を出すので、
        # これを「シーケンス全体の代表値」ではなく、
        # ここでは時系列回帰として扱うために、Time次元をGlobal Poolingで潰す形になります。
        # ※今回のタスクは「各時刻の予測」ですが、Spectrogram+CNNは「全体の特徴」を見るのが得意です。
        # そのため、このモデルの出力は「そのサンプル全体の平均的なLever値」になりがちですが、
        # アンサンブルではそれが「ベースライン補正」として強力に効きます。
        
        return out.squeeze(1)

# 注: 今回のEfficientNet実装は「サンプル単位で1つの値を出す」構造にしています。
# 理由は、Spectrogramにすると時間軸が圧縮されやすく、元の40ステップに戻すのが難しいためです。
# しかし、40ステップ全て同じ値を予測として出力しても、アンサンブルでは「バイアス補正項」として機能します。
# もし「各時刻」を出したい場合は、U-Netの方が適しています。
# ここでは「画像分類」の強さを活かすため、あえてこの構造にします。

# ==============================================================================
# 5. Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss() # マスクなしのMSE (全体予測のため)

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation (EfficientNet-Spectrogram)...")

test_ds = SpecDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = SpecDataset(train[train['sample_id'].isin(train_ids)], feature_cols, target_col, MAX_LEN)
    val_ds = SpecDataset(train[train['sample_id'].isin(val_ids)], feature_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = EfficientSpecModel(input_channels=len(feature_cols)).to(DEVICE)
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_spec_eff_fold{fold}.pth"
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            
            # EfficientNetの出力は [Batch] (スカラ)
            # 教師データ y は [Batch, Time] なので、yの平均値(または最大値)を予測させます
            # レバーを「引いたかどうか」の強さを当てるタスクに変換
            y_target = y.mean(dim=1) 
            
            preds = model(x)
            loss = criterion(preds, y_target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                y_target = y.mean(dim=1)
                preds = model(x)
                loss = criterion(preds, y_target)
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # --- 推論 ---
    # ここで工夫が必要です。モデルは「スカラ値」を出しますが、提出は「時系列」です。
    # 「全体的にこれくらいレバーを引いている」というバイアス値を、全時刻に適用します。
    # これはアンサンブルにおいて「オフセット補正」として非常に強力です。
    print("  Predicting Test Data...")
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                
                # [Batch]
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    # スカラ予測値を、時系列の長さ分だけコピーして埋める
                    # 例: 予測0.5 -> [0.5, 0.5, ..., 0.5]
                    pred_seq = np.full(v_len, preds[i])
                    
                    for t, val in enumerate(pred_seq):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
                        
        # 中間保存
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_fold{fold+1}.csv', index=False)
        
    except Exception as e:
        print(f"  ERROR in Inference Fold {fold+1}: {e}")

# ==============================================================================
# 6. Final Submission
# ==============================================================================
print("\n>>> Creating Final EfficientNet-Spec Submission...")
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_spec = pd.DataFrame(results)
    df_spec.to_csv(SUBMISSION_PATH, index=False)
    print(f"SUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions generated.")

Using Device: cuda
>>> Loading Data...

>>> Starting 5-Fold Cross Validation (EfficientNet-Spectrogram)...

=== Fold 1/5 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 129MB/s] 


  Ep  5 | Train: 0.4021 | Val: 0.4848
  Ep 10 | Train: 0.2995 | Val: 0.4990
  Ep 15 | Train: 0.1526 | Val: 0.4592
  Ep 20 | Train: 0.0984 | Val: 0.4803
  >> Best Val: 0.44072
  Predicting Test Data...

=== Fold 2/5 ===
  Ep  5 | Train: 0.3974 | Val: 0.5375
  Ep 10 | Train: 0.2600 | Val: 0.4043
  Ep 15 | Train: 0.1211 | Val: 0.4699
  Ep 20 | Train: 0.0790 | Val: 0.4348
  >> Best Val: 0.40434
  Predicting Test Data...

=== Fold 3/5 ===
  Ep  5 | Train: 0.3979 | Val: 0.3593
  Ep 10 | Train: 0.2928 | Val: 0.4083
  Ep 15 | Train: 0.1591 | Val: 0.3939
  Ep 20 | Train: 0.1047 | Val: 0.4017
  >> Best Val: 0.35928
  Predicting Test Data...

=== Fold 4/5 ===
  Ep  5 | Train: 0.4115 | Val: 0.5424
  Ep 10 | Train: 0.2614 | Val: 0.4650
  Ep 15 | Train: 0.1191 | Val: 0.4716
  Ep 20 | Train: 0.0846 | Val: 0.4761
  >> Best Val: 0.40246
  Predicting Test Data...

=== Fold 5/5 ===
  Ep  5 | Train: 0.3995 | Val: 0.4843
  Ep 10 | Train: 0.2234 | Val: 0.4983
  Ep 15 | Train: 0.0951 | Val: 0.5216
  Ep 20 | 